# 📈 Calculadora de Beta

Esta herramienta calcula el **Beta (β)** de cualquier acción contra un benchmark (ETF o índice) usando datos reales de Yahoo Finance.

### ¿Qué es el Beta?
El Beta mide la sensibilidad de una acción frente a los movimientos del mercado:
- **β > 1** → la acción es más volátil que el mercado
- **β = 1** → se mueve igual que el mercado  
- **β < 1** → es menos volátil (más defensiva)
- **β < 0** → se mueve en sentido contrario al mercado

Se obtiene a partir de la **regresión lineal** entre los retornos de la acción y los del benchmark:

$$R_{accion} = \alpha + \beta \cdot R_{benchmark} + \epsilon$$

---
**Tickers de ejemplo:**
- Acciones: `AAPL`, `MSFT`, `MELI`, `GGAL`, `BABA`
- Benchmarks: `SPY` (S&P500), `QQQ` (Nasdaq), `ARGT` (Argentina), `EWZ` (Brasil)

In [ ]:
# Instalación de librerías (solo necesario en Colab)
!pip install yfinance ipywidgets --quiet

In [ ]:
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')

In [ ]:
def obtener_retornos(ticker, periodo):
    """
    Descarga precios ajustados (Adj Close) de Yahoo Finance
    y calcula los retornos mensuales.
    """
    data = yf.download(ticker, period=periodo, interval='1mo', progress=False, auto_adjust=True)
    if data.empty:
        raise ValueError(f"No se encontraron datos para '{ticker}'. Verificá el ticker.")
    precios = data['Close'].squeeze()
    retornos = precios.pct_change().dropna()
    return retornos


def calcular_beta(retornos_accion, retornos_bench):
    """
    Calcula Beta, Alfa y R² usando regresión lineal (OLS).
    Beta = Cov(Ra, Rb) / Var(Rb)
    """
    df = pd.concat([retornos_accion, retornos_bench], axis=1).dropna()
    df.columns = ['accion', 'bench']

    x = df['bench'].values
    y = df['accion'].values

    # Regresión lineal manual (igual que en las entregas)
    beta, alpha = np.polyfit(x, y, 1)

    # R²
    y_pred = alpha + beta * x
    ss_res = np.sum((y - y_pred) ** 2)
    ss_tot = np.sum((y - np.mean(y)) ** 2)
    r2 = 1 - ss_res / ss_tot

    return beta, alpha, r2, df


def interpretar_beta(beta, ticker_accion, ticker_bench):
    if beta > 1.2:
        return f"🔴 Agresiva: {ticker_accion} es más volátil que {ticker_bench}. Por cada 1% que sube el benchmark, {ticker_accion} sube ~{beta:.2f}%."
    elif beta >= 0.8:
        return f"🟡 Neutral: {ticker_accion} se mueve de forma similar a {ticker_bench}. Riesgo de mercado cercano al promedio."
    elif beta >= 0:
        return f"🟢 Defensiva: {ticker_accion} es menos volátil que {ticker_bench}. Se mueve menos ante cambios del mercado."
    else:
        return f"🔵 Inversa: {ticker_accion} tiene correlación negativa con {ticker_bench}. Se mueve en dirección contraria."


def graficar(df, beta, alpha, r2, ticker_accion, ticker_bench):
    fig = plt.figure(figsize=(14, 10))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.35)

    # ── Gráfico 1: Dispersión con recta de regresión ──
    ax1 = fig.add_subplot(gs[0, :])
    ax1.scatter(df['bench'] * 100, df['accion'] * 100,
                color='steelblue', alpha=0.7, s=60, label='Retornos mensuales')

    x_line = np.linspace(df['bench'].min(), df['bench'].max(), 100)
    y_line = alpha + beta * x_line
    ax1.plot(x_line * 100, y_line * 100, color='tomato', linewidth=2,
             label=f'Regresión: y = {alpha*100:.2f}% + {beta:.2f}x')

    ax1.axhline(0, color='gray', linewidth=0.5, linestyle='--')
    ax1.axvline(0, color='gray', linewidth=0.5, linestyle='--')
    ax1.set_xlabel(f'Retorno mensual {ticker_bench} (%)', fontsize=11)
    ax1.set_ylabel(f'Retorno mensual {ticker_accion} (%)', fontsize=11)
    ax1.set_title(f'Regresión lineal: {ticker_accion} vs {ticker_bench}', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10)
    ax1.grid(True, alpha=0.3)

    # ── Gráfico 2: Retornos acumulados ──
    ax2 = fig.add_subplot(gs[1, :])
    cum_accion = (1 + df['accion']).cumprod() - 1
    cum_bench  = (1 + df['bench']).cumprod() - 1
    ax2.plot(cum_accion.index, cum_accion * 100, label=ticker_accion, color='steelblue', linewidth=2)
    ax2.plot(cum_bench.index,  cum_bench * 100,  label=ticker_bench,  color='gray',      linewidth=2, linestyle='--')
    ax2.set_ylabel('Retorno acumulado (%)', fontsize=11)
    ax2.set_title('Retornos acumulados', fontsize=13, fontweight='bold')
    ax2.legend(fontsize=10)
    ax2.grid(True, alpha=0.3)
    ax2.axhline(0, color='black', linewidth=0.5)

    plt.suptitle(f'Análisis de Beta — {ticker_accion} vs {ticker_bench}', fontsize=15, fontweight='bold', y=1.01)
    plt.show()


print('✅ Funciones definidas correctamente')

In [ ]:
# ── Interfaz interactiva ──

style = {'description_width': '160px'}
layout = widgets.Layout(width='350px')

w_accion = widgets.Text(
    value='AAPL',
    description='Acción (ticker):',
    style=style, layout=layout
)
w_bench = widgets.Text(
    value='SPY',
    description='Benchmark (ticker):',
    style=style, layout=layout
)
w_periodo = widgets.Dropdown(
    options=[('1 año', '1y'), ('2 años', '2y'), ('3 años', '3y'), ('5 años', '5y')],
    value='2y',
    description='Período:',
    style=style, layout=layout
)
btn = widgets.Button(
    description='Calcular Beta',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
output = widgets.Output()

def on_click(b):
    with output:
        clear_output(wait=True)
        ticker_accion = w_accion.value.strip().upper()
        ticker_bench  = w_bench.value.strip().upper()
        periodo       = w_periodo.value

        if not ticker_accion or not ticker_bench:
            print('⚠️  Ingresá ambos tickers.')
            return

        print(f'⏳ Descargando datos de {ticker_accion} y {ticker_bench}...')

        try:
            ret_accion = obtener_retornos(ticker_accion, periodo)
            ret_bench  = obtener_retornos(ticker_bench,  periodo)
            beta, alpha, r2, df = calcular_beta(ret_accion, ret_bench)

            print('\n' + '='*55)
            print(f'  RESULTADOS: {ticker_accion} vs {ticker_bench} ({periodo})')
            print('='*55)
            print(f'  Beta  (β):      {beta:.4f}')
            print(f'  Alfa  (α):      {alpha*100:.4f}%  mensual')
            print(f'  R²:             {r2:.4f}')
            print(f'  Observaciones:  {len(df)} meses')
            print('='*55)
            print(f'  {interpretar_beta(beta, ticker_accion, ticker_bench)}')
            print('='*55 + '\n')

            graficar(df, beta, alpha, r2, ticker_accion, ticker_bench)

        except Exception as e:
            print(f'\n❌ Error: {e}')
            print('   Verificá que el ticker exista en Yahoo Finance.')

btn.on_click(on_click)

display(
    widgets.HTML('<h3 style="margin-bottom:12px">📊 Calculadora de Beta</h3>'),
    w_accion,
    w_bench,
    w_periodo,
    btn,
    output
)

---
## 📝 Notas metodológicas

- Se usan **precios ajustados (Adj Close)** para incorporar dividendos y splits
- Los retornos son **mensuales** (intervalo `1mo`)
- El Beta se calcula por **OLS (mínimos cuadrados ordinarios)**: `β = Cov(Ra, Rb) / Var(Rb)`
- El **R²** indica qué porción de la varianza de la acción es explicada por el benchmark
- El **Alfa (α)** es el retorno mensual promedio no explicado por el mercado (Jensen's Alpha)

### Benchmarks sugeridos
| Ticker | Descripción |
|--------|-------------|
| `SPY`  | S&P 500 (EE.UU.) |
| `QQQ`  | Nasdaq 100 |
| `ARGT` | ETF de acciones argentinas |
| `EWZ`  | ETF de Brasil |
| `ILF`  | ETF de Latinoamérica |

---
*Proyecto desarrollado como parte del curso Finanzas I — Universidad de Montevideo*